# Genesis Manthan — Phase 1: SFT on Kaggle T4

**Runtime**: Kaggle T4 GPU (16GB VRAM)  
**Duration**: ~2–3 hours  
**Output**: `shahansha/Manthan-1.5B-sft` on HuggingFace Hub

**Before running**: Set `HF_TOKEN` secret in Kaggle Secrets (Settings → Add Secret).

This notebook is idempotent — safe to re-run from any cell.

In [ ]:
# 1. INSTALL DEPENDENCIES
# Run once per session — idempotent
# Let unsloth[kaggle-new] pull in its required transformers/trl versions.
# DO NOT pin transformers or trl — unsloth 2026.x requires transformers>=4.51.3 and trl>=0.18.2.
import subprocess, sys

# Remove Kaggle preinstalls that create resolver noise but are not used in this notebook.
# gcsfs/s3fs pin incompatible fsspec versions, and bigframes pulls optional BigQuery deps.
for pkg in ['bigframes', 'gcsfs', 's3fs']:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', pkg], capture_output=True)

# Install unsloth first so pip can resolve compatible transformers/trl.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth[kaggle-new]'], check=True)

# Install remaining packages without forcing incompatible transitive upgrades.
packages = [
    'datasets>=2.20.0',
    'peft>=0.12.0',
    'accelerate>=0.33.0',
    'huggingface_hub',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', pkg], check=True)

# Import unsloth before transformers/trl so all patches are active.
import unsloth  # must be imported before trl/transformers
import importlib

print(f'unsloth: {unsloth.__version__}')

# Print resolved versions for debugging
for lib in ['transformers', 'trl', 'peft']:
    try:
        mod = importlib.import_module(lib)
        print(f'{lib}: {mod.__version__}')
    except Exception as e:
        print(f'{lib}: FAILED — {e}')

print('✅ Dependencies installed')


In [ ]:
# 2. AUTHENTICATE WITH HUGGINGFACE HUB
import os
from huggingface_hub import login

# On Kaggle: use Kaggle Secrets (Settings > Add Secret > Name: HF_TOKEN)
# Locally: set HF_TOKEN environment variable
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    raise ValueError('HF_TOKEN not found. Set it in Kaggle Secrets or as an environment variable.')

login(token=hf_token, add_to_git_credential=False)
print('✅ Authenticated with HuggingFace Hub')

In [ ]:
# 3. VERIFY ENVIRONMENT
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')
    # T4 does not support bfloat16 — enforce float16
    assert not torch.cuda.is_bf16_supported() or True, 'Verify float16 is used'
    print('dtype: torch.float16 (bfloat16 excluded for T4 compatibility)')
else:
    print('⚠️ No CUDA found — running on CPU (slow, for smoke-test only)')

# Detect Kaggle
is_kaggle = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
print(f'Kaggle env: {is_kaggle}')

In [ ]:
# 4. LOAD MODEL WITH UNSLOTH
from unsloth import FastLanguageModel

MODEL_ID = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
)

if hasattr(model, 'generation_config') and model.generation_config is not None:
    model.generation_config.max_length = None

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()
print('✅ Model loaded and LoRA applied')

In [ ]:
# 5. LOAD DATASET
from datasets import load_dataset

# Load from HuggingFace Hub (published in project setup)
# Falls back to local processed data if Hub dataset not yet uploaded
try:
    dataset = load_dataset('shahansha/manthan-tool-reasoning-v1')
    print(f'Loaded from Hub: {dataset}')
except Exception:
    # Load from Kaggle Dataset input
    import json, pathlib
    data_path = pathlib.Path('/kaggle/input/manthan-tool-reasoning-v1')
    if data_path.exists():
        dataset = load_dataset('json', data_files={
            'train': str(data_path / 'train.jsonl'),
            'validation': str(data_path / 'validation.jsonl'),
        })
        print(f'Loaded from Kaggle dataset: {dataset}')
    else:
        raise FileNotFoundError('Dataset not found. Upload manthan-tool-reasoning-v1 as a Kaggle dataset.')

print(f'Train examples: {len(dataset["train"])}')
print(f'Val examples: {len(dataset.get("validation", dataset.get("test", [])))}')

# Verify format
sample = dataset['train'][0]
print(f'\nSample keys: {list(sample.keys())}')
print(f'Text preview: {sample["text"][:200]}...')

In [ ]:
# 6. RUN SFT TRAINING
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import HfApi

# Verify token has write access before starting (saves time vs failing mid-training)
api = HfApi()
whoami = api.whoami(token=hf_token)
hf_username = whoami['name']          # exact casing from HF (e.g. 'Shahansha')
token_type = whoami.get('auth', {}).get('accessToken', {}).get('role', 'unknown')
print(f'HF user: {hf_username} | token role: {token_type}')

if token_type not in ('write', 'admin'):
    raise RuntimeError(
        f"Token role is '{token_type}' — write access required.\n"
        "Go to https://huggingface.co/settings/tokens → New token → Role: Write\n"
        "Then update HF_TOKEN in Kaggle Secrets (Settings → Add-on → Secrets)."
    )
print('✅ Token verified — write access confirmed')

# Use exact username casing to avoid 403 on repo creation
HUB_MODEL_ID = f'{hf_username}/Manthan-1.5B-sft'
print(f'Hub target: {HUB_MODEL_ID}')

training_args = TrainingArguments(
    output_dir='/kaggle/working/manthan-sft',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=100,
    fp16=True,
    bf16=False,          # CRITICAL: T4 does not support bfloat16
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    eval_strategy='steps',
    eval_steps=200,
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_token=hf_token,
    hub_strategy='checkpoint',
    report_to='none',
    seed=42,
    # Workaround for Unsloth bug: newer transformers defaults average_tokens_across_devices=True,
    # which does loss *= factor in-place on Unsloth's fused loss tensor, converting it to int.
    # Setting False disables that scaling and keeps loss as a tensor. (unsloth/unsloth#3769)
    average_tokens_across_devices=False,
)

# Resolve eval split — carve one out of train if no validation/test split exists
if 'validation' in dataset:
    train_ds = dataset['train']
    eval_ds = dataset['validation']
elif 'test' in dataset:
    train_ds = dataset['train']
    eval_ds = dataset['test']
else:
    # No separate eval split — hold out 5% of training data
    split = dataset['train'].train_test_split(test_size=0.05, seed=42)
    train_ds = split['train']
    eval_ds = split['test']

print(f'Train examples: {len(train_ds)}')
print(f'Eval examples:  {len(eval_ds)}')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

print(f'Training on {len(train_ds)} examples...')
print(f'Will push checkpoints to: {HUB_MODEL_ID}')
trainer.train()


In [ ]:
# 7. SAVE AND PUSH FINAL MODEL
model.save_pretrained('/kaggle/working/manthan-sft-final')
tokenizer.save_pretrained('/kaggle/working/manthan-sft-final')

# Push final merged model
model.push_to_hub_merged(
    HUB_MODEL_ID,
    tokenizer=tokenizer,
    save_method='merged_16bit',
)
print(f'✅ Final model pushed to: https://huggingface.co/{HUB_MODEL_ID}')

In [ ]:
# 8. QUICK INFERENCE TEST
FastLanguageModel.for_inference(model)

messages = [
    {'role': 'system', 'content': 'You are Genesis Manthan. Always solve by calling tools.'},
    {'role': 'user', 'content': 'What is the square root of 144?'},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to('cuda')

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=True,
)
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Model response:')
print(response)

# Check for tool call
if '<tool_call>' in response:
    print('\n✅ Model generated tool call correctly')
else:
    print('\n⚠️ No tool call generated — may need more training or adjusted system prompt')

## Next Step: GRPO Training

After SFT completes and the model is pushed to `shahansha/Manthan-1.5B-sft`, proceed to:
**`03_grpo_kaggle.ipynb`** for Phase 2 GRPO training with tool-execution rewards.

Expected SFT improvements:
- Tool call parsability: ~7% → ~50%
- GSM8K: ~45% → ~55%
- MBPP: ~35% → ~42%